# NightWatch - Rough Sleeper Survival Risk Intelligence System
### City of Melbourne Open Data | Data Science Capstone | T1 2026

**Student:** Manya Mahajan (223222623)
**Use Case ID:** UC00210
**Datasets:** Pedestrian Counting System · Microclimate Sensors · Street Furniture · CLUE

---

## Background

Melbourne's CBD has a persistent and visible rough sleeping population. People sleeping on Melbourne's streets face life-threatening conditions during extreme heat events and cold winter nights and those risks are largely preventable if outreach workers can be deployed to the right locations at the right time.

The challenge is not a lack of care. It is a lack of spatial intelligence. Outreach services currently operate reactively, responding after someone is already in crisis. There is no data-driven system that identifies where risk is accumulating *before* it becomes an emergency.

NightWatch proposes to close that gap using four openly available City of Melbourne datasets to build a **Survival Risk Surface**- a precinct-level scoring system that identifies which CBD locations combine the most dangerous overnight conditions for rough sleepers.

---

## Use Case

| Component | Detail |
|---|---|
| **Product** | A Survival Risk Surface - a spatial scoring tool identifying high-risk precincts for overnight rough sleeper exposure |
| **Primary Actors** | City of Melbourne Outreach Teams, Public Health Officers, Council Planning Staff |
| **Secondary Actors** | Rough sleepers, community support organisations, general public |
| **Scenario** | An outreach worker needs to know which CBD locations to prioritise on a given night based on temperature, isolation, shelter availability, and warmth refuge proximity |
| **Goal** | Generate a nightly risk score per precinct so outreach workers leave the depot knowing exactly where to go -shifting from reactive to preventive deployment |

---

## User Stories

> **As an** outreach worker for the City of Melbourne,
> **I want** to know which CBD locations have the highest combination of dangerous temperatures, pedestrian isolation, and no nearby shelter or warmth refuges tonight,
> **in order to** prioritise my route and reach the most vulnerable rough sleepers before their situation becomes life-threatening.

> **As a** public health officer,
> **I want** to understand which precincts consistently experience dangerous overnight temperature and isolation conditions,
> **in order to** recommend targeted infrastructure investment - additional seating, shelter, and 24-hour warmth refuges in the highest-risk areas.

> **As a** City of Melbourne urban planner,
> **I want** to identify gaps in shelter and food & beverage access in high foot-traffic overnight precincts,
> **in order to** inform planning permit conditions and capital works priorities that improve safety for Melbourne's most vulnerable residents.

---

## Datasets Used

| # | Dataset | Portal Link | Role in NightWatch |
|---|---|---|---|
| 1 | Pedestrian Counting System | [Link](https://data.melbourne.vic.gov.au/explore/dataset/pedestrian-counting-system-monthly-counts-per-hour) | Identifies which locations are isolated overnight |
| 2 | Microclimate Sensors | [Link](https://data.melbourne.vic.gov.au/explore/dataset/microclimate-sensors-data) | Captures street-level temperature and wind speed |
| 3 | Street Furniture | [Link](https://data.melbourne.vic.gov.au/explore/dataset/street-furniture-including-bollards-bicycle-rails-bins-drinking-fountains-horse-) | Maps seat and shelter locations across the CBD |
| 4 | CLUE - Business Establishments | [Link](https://data.melbourne.vic.gov.au/explore/dataset/business-establishments-per-block-by-clue-industry) | Identifies food & beverage businesses as warmth refuges |

---

## Workflow

1. **Environment Setup & Data Loading** — Import libraries, configure API access, load all four datasets
2. **Data Exploration** — Understand structure, column types, and scale of each dataset
3. **Data Cleaning & Preprocessing** — Filter to NightWatch scope, handle missing values
4. **Exploratory Data Analysis (Week 4)** — Descriptive statistics, missing value analysis, initial visualisations
5. **Deep Exploratory Analysis (Week 5)** — Spatial and temporal risk visualisations, cross-dataset patterns

## Section 1: Environment Setup

We begin by importing all libraries required for data analysis, visualisation, and API access. Several essential packages are used throughout this analysis:

- **pandas & numpy** - data manipulation and numerical computation
- **matplotlib & seaborn** - static visualisation and statistical charting
- **requests** - API calls to the City of Melbourne Open Data Portal

In [3]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import requests
import warnings
import getpass
from io import StringIO

# Configuration 
warnings.filterwarnings('ignore')
plt.style.use('default')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("NIGHTWATCH — ROUGH SLEEPER SURVIVAL RISK INTELLIGENCE SYSTEM")
print("=" * 70)
print("City of Melbourne Open Data | Data Science Capstone | T1 2026")
print("=" * 70)

NIGHTWATCH — ROUGH SLEEPER SURVIVAL RISK INTELLIGENCE SYSTEM
City of Melbourne Open Data | Data Science Capstone | T1 2026


### Section 1: Output Analysis

Libraries loaded successfully. The environment is configured with warning suppression, default plot styling, and pandas display settings optimised for wide datasets. All packages required for data retrieval, analysis, and visualisation are available and ready.

## Section 2: Data Collection Function

All datasets are accessed directly via the **City of Melbourne Open Data API v2.1** -no local files are used, ensuring the analysis is fully replicable. The City of Melbourne datasets are publicly accessible and the code is safe to share.

A reusable `collect_data()` function is defined to fetch any dataset using its portal ID. It handles pagination, errors, and decoding gracefully and provides clear feedback on loading success.

In [6]:
def collect_data(dataset_id, limit=10000):
    """
    Fetch a dataset from the City of Melbourne Open Data API v2.1.

    Parameters:
        dataset_id : str - dataset identifier from the portal URL
        limit      : int - maximum number of records to fetch

    Returns:
        pd.DataFrame or None
    """
    base_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/"
    url = f"{base_url}{dataset_id}/exports/csv"

    params = {
        "select"  : "*",
        "limit"   : limit,
        "lang"    : "en",
        "timezone": "Australia/Melbourne"
    }

    try:
        response = requests.get(url, params=params)
        if response.status_code == 200:
            content = response.content.decode("utf-8")
            df = pd.read_csv(StringIO(content), delimiter=";")
            print(f" Loaded '{dataset_id}': {df.shape[0]:,} rows × {df.shape[1]} columns")
            return df
        else:
            print(f" Failed '{dataset_id}'. Status code: {response.status_code}")
            return None
    except Exception as e:
        print(f" Error loading '{dataset_id}': {str(e)}")
        return None

print("Data collection function defined successfully")

Data collection function defined successfully


### Section 2: Output Analysis

The `collect_data()` function was defined successfully using the City of Melbourne Open Data API v2.1 endpoint. The function handles CSV decoding, semicolon delimiters, error handling, and provides clear loading feedback making it safe and straightforward to share and reproduce.


## Section 3: Data Loading

We now load all four NightWatch datasets directly from the City of Melbourne Open Data Portal. Each dataset has a distinct role in building the Survival Risk Surface:

- **Pedestrian Counting System** - hourly counts from sensors across the CBD, used to identify isolated overnight locations
- **Microclimate Sensors** - street-level temperature, humidity, and wind speed readings every 15 minutes
- **Street Furniture** - location and condition of all seats, shelters, and public assets across the CBD
- **CLUE (Business Establishments)** - census of all business types per block, used to identify food & beverage warmth refuges

Note: The microclimate dataset uses the historical ID `microclimate-sensors-data` rather than the live feed `microclimate-sensor-readings`, which only returns the most recent snapshot.

In [ ]:
print("\n LOADING DATASETS")
print("-" * 50)

# Dataset IDs from data.melbourne.vic.gov.au 
PEDESTRIAN_ID    = "pedestrian-counting-system-monthly-counts-per-hour"
MICROCLIMATE_ID  = "microclimate-sensors-data"
FURNITURE_ID     = "street-furniture-including-bollards-bicycle-rails-bins-drinking-fountains-horse-"
CLUE_ID          = "business-establishments-per-block-by-clue-industry"

pedestrian       = collect_data(PEDESTRIAN_ID,     limit=50000)
microclimate     = collect_data(MICROCLIMATE_ID,   limit=100000)
street_furniture = collect_data(FURNITURE_ID,      limit=5000)
clue             = collect_data(CLUE_ID,           limit=10000)

# Verify all datasets loaded
datasets = {
    "Pedestrian Counting" : pedestrian,
    "Microclimate Sensors": microclimate,
    "Street Furniture"    : street_furniture,
    "CLUE"                : clue
}

missing = [name for name, df in datasets.items() if df is None]
if missing:
    print(f"\n WARNING: Failed to load: {', '.join(missing)}")
else:
    print("\n All datasets loaded successfully!")


 LOADING DATASETS
--------------------------------------------------


### Section 3: Output Analysis

All four datasets loaded successfully from the City of Melbourne Open Data API v2.1.

| Dataset | Rows | Columns | Note |
|---|---|---|---|
| Pedestrian Counting | 50,000 | 9 | Capped at limit - full dataset is significantly larger |
| Microclimate Sensors | 100,000 | 16 | Historical dataset with rich environmental readings |
| Street Furniture | 5,000 | 15 | Includes all asset types across the CBD |
| CLUE | 10,000 | 24 | 2024 census data- most current available |

All datasets loaded without errors. The microclimate dataset correctly uses the historical ID `microclimate-sensors-data` rather than the live feed, which would only return a single snapshot of 56 rows.


## Section 4: Data Exploration

Before any cleaning or analysis, we inspect the structure of each dataset - column names, data types, shape, and sample rows. This step is essential to understand what variables are available, how they are formatted, and how they relate to the NightWatch use case before any transformations are applied.

In [ ]:
for name, df in datasets.items():
    print("=" * 70)
    print(f" {name.upper()}")
    print("=" * 70)
    print(f"Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"Columns : {list(df.columns)}")
    print(f"\nData Types:\n{df.dtypes}")
    print(f"\nFirst 2 rows:\n{df.head(2)}")
    print()

### Section 4: Output Analysis

**Pedestrian Counting** - 9 columns covering sensor ID, date, hour of day, directional counts, total pedestrian count, sensor name, and location as a combined coordinate string. `sensing_date` is stored as object and will require datetime conversion. `hourday` (0–23) is the key filter column for overnight scoping.

**Microclimate Sensors** - 16 columns including device ID, timestamp, sensor location name, combined lat/long, wind direction (min/avg/max), wind speed, air temperature, relative humidity, atmospheric pressure, PM2.5, PM10, and noise. `airtemperature` and `latlong` are the primary columns for NightWatch analysis.

**Street Furniture** - 15 columns including asset class, asset type, model description, location description, condition rating (1–5), easting/northing coordinates, and a combined coordinate field. The `asset_type` column is the key filter for isolating seats from bollards, bins, and other assets.

**CLUE** - 24 columns covering census year, block ID, suburb, and one column per CLUE industry category showing establishment counts per block. `food_and_beverage_services` is the primary column for identifying warmth refuge candidates.


## Section 5: Data Cleaning & Preprocessing

With all four datasets confirmed, we now clean and filter each one to match the NightWatch use case scope. The following transformations are applied:

- **Pedestrian** - filter to late-night hours (10pm–5am), the period of highest rough sleeper exposure; convert date column to datetime
- **Microclimate** - convert timestamps to datetime; drop rows with null temperature or location values which indicate sensor outages
- **Street Furniture** - filter to seats only, removing bollards, bins, and other assets irrelevant to shelter analysis
- **CLUE** - filter to blocks containing at least one food & beverage establishment, retaining only potential warmth refuge locations

In [ ]:
# 1. PEDESTRIAN — late-night filter (10pm–5am) 
pedestrian['sensing_date'] = pd.to_datetime(pedestrian['sensing_date'])
pedestrian_night = pedestrian[pedestrian['hourday'].isin([22, 23, 0, 1, 2, 3, 4, 5])]

print("PEDESTRIAN — Late-night filter")
print(f"  Before : {pedestrian.shape[0]:,} rows")
print(f"  After  : {pedestrian_night.shape[0]:,} rows")
print(f"  Hours  : {sorted(pedestrian_night['hourday'].unique())}\n")

# 2. MICROCLIMATE - datetime conversion & null removal 
microclimate['received_at'] = pd.to_datetime(microclimate['received_at'])
microclimate_clean = microclimate.dropna(subset=['airtemperature', 'latlong'])

print("MICROCLIMATE — Clean temperature readings")
print(f"  Before : {microclimate.shape[0]:,} rows")
print(f"  After  : {microclimate_clean.shape[0]:,} rows\n")

# 3. STREET FURNITURE - seats only 
print("Street furniture asset types:")
print(f"{street_furniture['asset_type'].value_counts()}\n")

seats = street_furniture[street_furniture['asset_type'] == 'Seat'].copy()

print("STREET FURNITURE — Seats only")
print(f"  Before : {street_furniture.shape[0]:,} rows")
print(f"  After  : {seats.shape[0]:,} rows\n")

# 4. CLUE - food & beverage blocks only 
clue_clean = clue[clue['food_and_beverage_services'] > 0].copy()

print("CLUE - Blocks with food & beverage businesses")
print(f"  Before : {clue.shape[0]:,} rows")
print(f"  After  : {clue_clean.shape[0]:,} rows")

### Section 5: Output Analysis

| Dataset | Before | After | Filter Applied |
|---|---|---|---|
| Pedestrian | 50,000 rows | 15,180 rows | Late-night hours only (10pm–5am) |
| Microclimate | 100,000 rows | 95,635 rows | Null temperature and location rows removed |
| Street Furniture | 5,000 rows | 701 rows | Seats only (from 12 asset types) |
| CLUE | 10,000 rows | 4,764 rows | Blocks with at least one food & beverage establishment |

Approximately 30% of all pedestrian records fall within the overnight window indicating meaningful late-night activity in the dataset. The microclimate drop of ~4,400 rows reflects normal IoT sensor outages. Seats account for only 14% of all street furniture- the majority being bollards and bicycle rails which are irrelevant to shelter analysis.



## Section 6: Exploratory Data Analysis - Week 4

### 6.1 Descriptive Statistics

We begin EDA by computing descriptive statistics across all four cleaned datasets. This gives us a quantitative baseline - understanding the central tendency, spread, and extreme values of the key variables before any visualisation. For NightWatch, the most critical variables are:

- `pedestriancount` - how many people are present overnight
- `airtemperature` - thermal exposure risk
- `condition_rating` - usability of available seating
- `food_and_beverage_services` - density of warmth refuges per block

In [ ]:
print("=" * 70)
print(" DESCRIPTIVE STATISTICS")
print("=" * 70)

print("\n── Pedestrian (Late-night) ──")
print(pedestrian_night[['hourday', 'pedestriancount']].describe().round(2))

print("\n── Microclimate (Clean) ──")
print(microclimate_clean[['airtemperature', 'relativehumidity', 'averagewindspeed']].describe().round(2))

print("\n── Street Furniture (Seats) ──")
print(seats[['condition_rating']].describe().round(2))

print("\n── CLUE (Food & Beverage Blocks) ──")
print(clue_clean[['food_and_beverage_services', 'total_establishments_in_block']].describe().round(2))

### 6.1 Output Analysis - Descriptive Statistics

**Pedestrian (Late-night)**
- Average of 95 pedestrians per hourly count but a high standard deviation of 220  indicating most locations are genuinely quiet overnight while a small number of hotspots drive the mean upward
- Maximum count of 5,045 in a single hour confirms locations like Swanston Street remain heavily trafficked even past midnight
- Median of just 26 is the most telling figure - most overnight sensor readings are critically low, representing exactly the isolation conditions NightWatch targets

**Microclimate**
- Average temperature of 16.2°C across all readings, with a minimum of -0.8°C and a maximum of 40.7°C, confirming the dataset captures genuinely dangerous extremes at both ends of the thermal spectrum
- Average wind speed of 1.05 km/h is low overall, however the maximum of 10.4 km/h combined with low temperatures creates dangerous wind chill conditions for rough sleepers
- Relative humidity ranges from 6.7% to 99.8%, adding a further comfort and health dimension to the risk profile

**Street Furniture (Seats)**
- Average condition rating of 3.39 out of 5 - the majority of seats are in acceptable condition
- Minimum of 1.0 indicates some seats are in very poor condition and would not provide usable or safe shelter
- Standard deviation of 0.81 suggests relatively consistent ratings, with most seats clustered between 3 and 4

**CLUE (Food & Beverage)**
- Maximum of 109 food & beverage establishments in a single block - these densely serviced blocks represent the strongest warmth refuge candidates for overnight outreach
- Median of just 3 establishments per block confirms that warmth refuge access is sparse across most of the CBD, with density concentrated in a small number of precincts



### 6.2 Missing Values Analysis

Understanding where data is missing is critical before building any risk model. Missing values in key columns such as temperature or location could silently distort the Survival Risk Surface if not identified and handled appropriately. We check each cleaned dataset for null values, calculate the percentage missing per column, and assess whether the gaps are in critical or non-critical columns.

In [ ]:
print("=" * 70)
print(" MISSING VALUES ANALYSIS")
print("=" * 70)

for name, df in [("Pedestrian Night", pedestrian_night),
                 ("Microclimate Clean", microclimate_clean),
                 ("Seats", seats),
                 ("CLUE Food & Beverage", clue_clean)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    pct = (missing / len(df) * 100).round(1)
    print(f"\n{name} ({len(df):,} rows):")
    if missing.empty:
        print("  No missing values ")
    else:
        for col in missing.index:
            print(f"  {col}: {missing[col]:,} missing ({pct[col]}%)")

### 6.2 Output Analysis- Missing Values

**Pedestrian Night** - No missing values. The dataset is complete across all 15,180 overnight rows, making it fully reliable for isolation analysis without any imputation required.

**Microclimate Clean** - Missing values are confined to wind-related columns (minimumwinddirection, maximumwinddirection, minimumwindspeed, gustwindspeed) at approximately 10.3% each, and particulate matter and noise columns (PM2.5, PM10, noise) at approximately 4.7% each. These gaps are likely due to sensors not equipped with wind instruments or air quality sensors at all locations. Critically, `airtemperature` - the primary NightWatch variable has no missing values following the cleaning step.

**Seats** - Missing values are minor and confined to descriptive columns only: `model_descr` (2.3%), `model_no` (0.3%), `location_desc` (0.3%), and `division`/`company` (0.1% each). Coordinate fields and asset type — the two columns required for spatial analysis are fully intact.

**CLUE Food & Beverage** - No missing values. The dataset is complete across all 4,764 filtered rows.

Overall the data quality across all four datasets is high. No critical columns contain missing values that would compromise the NightWatch Survival Risk Surface.



### 6.3 Visualisation 1 - Late-Night Pedestrian Activity by Hour

To understand the temporal pattern of overnight pedestrian exposure, we plot the average pedestrian count per hour across all late-night sensor readings. This chart reveals when the CBD transitions from active to critically isolated - the window when rough sleepers face the greatest risk of being undetected and unsupported.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hourly_avg = (
    pedestrian_night
    .groupby('hourday')['pedestriancount']
    .mean()
    .reindex([22, 23, 0, 1, 2, 3, 4, 5])
)
colors = ['#D94A4A' if v == hourly_avg.max() else '#4A90D9' for v in hourly_avg.values]
axes[0].bar(
    ['10pm','11pm','12am','1am','2am','3am','4am','5am'],
    hourly_avg.values,
    color=colors, edgecolor='white'
)
axes[0].set_title('Average Pedestrian Count by Hour (10pm–5am)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Hour of Night', fontsize=11)
axes[0].set_ylabel('Average Pedestrian Count', fontsize=11)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

axes[1].hist(
    pedestrian_night['pedestriancount'],
    bins=40, color='#4A90D9', edgecolor='white'
)
axes[1].axvline(
    pedestrian_night['pedestriancount'].median(),
    color='#D94A4A', linestyle='--', linewidth=2,
    label=f"Median: {pedestrian_night['pedestriancount'].median():.0f}"
)
axes[1].set_title('Distribution of Late-Night Pedestrian Counts', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Pedestrian Count per Hour', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### 6.3 Output Analysis - Late-Night Pedestrian Activity

**Average count by hour**
- 10pm (hour 22) records the highest average pedestrian count at approximately 275 - this is the tail end of Melbourne's evening economy with restaurants, bars, and entertainment still active
- Counts drop sharply after midnight, reaching their lowest point between 3am and 4am with averages of roughly 18–20 pedestrians - the most dangerous isolation window for rough sleepers
- A slight uptick at 5am reflects early morning commuters beginning to appear, offering some natural passive surveillance from 5am onwards
- 10pm is highlighted in red as the peak hour - the busiest overnight period where rough sleepers may be present but hidden in crowd activity

**Distribution of counts**
- The histogram is heavily right-skewed - the vast majority of hourly readings fall below 200 pedestrians, confirming that isolation is the norm across most sensor locations overnight
- The red dashed median line at 26 pedestrians underscores how quiet most overnight locations are - a person sleeping rough in most CBD locations after 2am is statistically alone



### 6.4 Visualisation 2 - Temperature Distribution Across Microclimate Sensors

To understand the thermal risk landscape across the CBD, we examine the distribution of air temperature readings. Rather than a simple histogram, we plot the full distribution using a kernel density estimate overlaid on a histogram, and add threshold markers for dangerous heat (≥35°C) and cold (≤5°C) - the conditions most harmful to people sleeping rough overnight.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

temps = microclimate_clean['airtemperature'].dropna()

axes[0].hist(temps, bins=50, density=True, alpha=0.4, color='#888888', edgecolor='white', label='Readings')
temps.plot.kde(ax=axes[0], color='#333333', linewidth=2, label='Density')
axes[0].axvspan(temps.min(), 5, alpha=0.15, color='#4A90D9', label='Cold risk (≤5°C)')
axes[0].axvspan(35, temps.max(), alpha=0.15, color='#D94A4A', label='Heat risk (≥35°C)')
axes[0].axvline(temps.mean(), color='#F5A623', linestyle='--', linewidth=1.5, label=f'Mean: {temps.mean():.1f}°C')
axes[0].set_title('Air Temperature Distribution\nwith Survival Risk Thresholds', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Temperature (°C)', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# ensure datetime before extracting month
microclimate_clean['received_at'] = pd.to_datetime(microclimate_clean['received_at'], utc=True)
microclimate_clean['month'] = microclimate_clean['received_at'].dt.month

monthly_temp = microclimate_clean.groupby('month')['airtemperature'].mean()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
bar_colors = ['#D94A4A' if v >= 22 else '#4A90D9' if v <= 10 else '#888888' for v in monthly_temp.values]

axes[1].bar(
    [month_labels[m-1] for m in monthly_temp.index],
    monthly_temp.values,
    color=bar_colors, edgecolor='white'
)
axes[1].axhline(35, color='#D94A4A', linestyle='--', linewidth=1, alpha=0.6, label='Heat threshold (35°C)')
axes[1].axhline(5, color='#4A90D9', linestyle='--', linewidth=1, alpha=0.6, label='Cold threshold (5°C)')
axes[1].set_title('Average Monthly Temperature\nAcross All Microclimate Sensors', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Month', fontsize=11)
axes[1].set_ylabel('Average Temperature (°C)', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f"Readings above 35°C (heat risk): {(temps >= 35).sum():,} ({(temps >= 35).mean()*100:.1f}%)")
print(f"Readings below 5°C  (cold risk): {(temps <= 5).sum():,} ({(temps <= 5).mean()*100:.1f}%)")

### 6.4 Output Analysis - Temperature Distribution

**Temperature distribution with risk thresholds**
- The KDE curve reveals a roughly bell-shaped distribution centred between 15–20°C, consistent with Melbourne's temperate climate
- The blue shaded cold risk zone (≤5°C) is small but present, confirming genuine winter cold exposure at street level across the sensor network
- The red shaded heat risk zone (≥35°C) captures a meaningful tail, these readings represent dangerous summer heat events that pose acute risk for rough sleepers without shade or water access
- The orange mean line at 16.2°C sits firmly in the moderate range, but the distribution's tails are what matter for NightWatch, extreme readings on both ends create life-threatening conditions

**Monthly average temperature**
- Summer months (December–February) show average temperatures in the 22–24°C range, coloured red - these are the primary heat risk months for rough sleepers
- Winter months (June–August) drop to around 9–11°C average, coloured blue - cold risk period where wind chill compounds exposure
- Neither the 35°C heat threshold nor the 5°C cold threshold is breached by monthly averages, confirming that risk is driven by daily extremes rather than sustained averages - reinforcing the need for real-time rather than seasonal risk assessment

**Risk summary**
- Readings above 35°C (heat risk): present across multiple sensors during summer months
- Readings below 5°C (cold risk): present across multiple sensors during winter months


## Section 7: Deep Exploratory Analysis - Week 5

Building on the statistical summaries from Week 4, we now move into deeper exploratory analysis using more sophisticated visualisations. The goal is to uncover spatial and temporal patterns across all four datasets that directly inform the NightWatch Survival Risk Surface  identifying which CBD locations experience the most dangerous combination of pedestrian isolation, extreme temperatures, poor shelter access, and limited warmth refuge availability overnight.

### 7.1 Visualisation 1 - Late-Night Pedestrian Activity Heatmap

This heatmap plots average pedestrian counts for the top 15 busiest overnight sensor locations against each hour of the night. It reveals two critical patterns for NightWatch: which locations remain active late into the night (higher risk of undetected rough sleepers), and which locations experience near-complete isolation in the early hours (3am–5am) when help is hardest to access.

In [ ]:
# Visualisation 1: Late-Night Pedestrian Activity Heatmap 
heatmap_data = (
    pedestrian_night
    .groupby(['sensor_name', 'hourday'])['pedestriancount']
    .mean()
    .unstack()
    .reindex(columns=[22, 23, 0, 1, 2, 3, 4, 5])
)

top_sensors = heatmap_data.sum(axis=1).nlargest(15).index
heatmap_data = heatmap_data.loc[top_sensors]

fig, ax = plt.subplots(figsize=(14, 8))

sns.heatmap(
    heatmap_data,
    cmap='YlOrRd',
    linewidths=0.5,
    linecolor='white',
    annot=True,
    fmt='.0f',
    cbar_kws={'label': 'Average Pedestrian Count'},
    ax=ax
)

ax.set_title(
    'Late-Night Pedestrian Activity Heatmap\nAverage Hourly Count by Sensor Location (10pm–5am)',
    fontsize=14, fontweight='bold', pad=15
)
ax.set_xlabel('Hour of Night', fontsize=12)
ax.set_ylabel('Sensor Location', fontsize=12)
ax.set_xticklabels(['10pm','11pm','12am','1am','2am','3am','4am','5am'], fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=9, rotation=0)

plt.tight_layout()
plt.show()

print("Top 3 busiest overnight locations:")
print(heatmap_data.sum(axis=1).nlargest(3).reset_index().to_string(index=False))
print("\nTop 3 most isolated overnight locations:")
print(heatmap_data.sum(axis=1).nsmallest(3).reset_index().to_string(index=False))

### 7.1 Output Analysis - Late-Night Pedestrian Activity Heatmap

The heatmap reveals clear and actionable temporal and spatial patterns in overnight pedestrian activity across the top 15 CBD sensor locations.

- **Swa123_T and Swa295_T** (Swanston Street sensors) record the highest activity at 10pm with counts of 1,405 and 1,019 respectively, confirming Swanston Street as Melbourne's busiest late-night corridor and a location where rough sleepers may be present but overlooked in the evening crowd
- **Activity drops sharply after midnight** across all locations - most sensors fall below 150 by 12am and below 100 by 1am, indicating the entire CBD transitions to a critically isolated state in the early hours
- **Swa31 and Eli368_T** show blank cells between 2am–4am - these sensors recorded zero pedestrian activity during those hours, representing complete isolation and the highest risk window for rough sleepers with no passive community surveillance
- **SwaCs_T** shows the steepest single-location decline from 952 at 10pm to just 3 at 4am making this precinct a priority target for NightWatch outreach deployment in the pre-dawn hours
- The colour gradient tells the NightWatch story clearly: the city transitions from warm red at 10pm to pale yellow by 3am -this is not a localised pattern but a systemic, city-wide collapse in overnight pedestrian presence

**Top 3 busiest overnight locations:** Swa295_T, Swa123_T, Swa31 — all Swanston Street corridor sensors

**Top 3 most isolated overnight locations:** Eli250_T, Eli368_T, SwaCs_T — these represent the highest-priority NightWatch deployment zones



### 7.2 Visualisation 2 - Thermal Risk Profile Across Sensor Locations

Rather than a simple temperature distribution, this visualisation examines the **thermal risk profile** of each sensor location, specifically how often each records temperatures in the dangerous heat range (≥35°C) or dangerous cold range (≤5°C). The stacked bar chart shows the proportion of each location's readings that fall into risk zones, while the dot plot reveals the full min-average-max temperature range per location. Together, these charts identify which CBD locations impose the greatest thermal burden on people sleeping rough.

In [ ]:
HEAT_THRESHOLD = 35
COLD_THRESHOLD = 5

risk_profile = (
    microclimate_clean
    .groupby('sensorlocation')['airtemperature']
    .agg(
        total_readings='count',
        heat_risk=lambda x: (x >= HEAT_THRESHOLD).sum(),
        cold_risk=lambda x: (x <= COLD_THRESHOLD).sum(),
        avg_temp='mean',
        max_temp='max',
        min_temp='min'
    )
    .reset_index()
)

risk_profile['heat_risk_pct'] = (risk_profile['heat_risk'] / risk_profile['total_readings'] * 100).round(1)
risk_profile['cold_risk_pct'] = (risk_profile['cold_risk'] / risk_profile['total_readings'] * 100).round(1)
risk_profile['total_risk_pct'] = risk_profile['heat_risk_pct'] + risk_profile['cold_risk_pct']

top_risk = risk_profile.nlargest(12, 'total_risk_pct').copy()
top_risk['short_name'] = top_risk['sensorlocation'].str.split(' - ').str[0].str[:30]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].barh(top_risk['short_name'], top_risk['cold_risk_pct'],
             color='#4A90D9', label='Cold risk (≤5°C)', height=0.6)
axes[0].barh(top_risk['short_name'], top_risk['heat_risk_pct'],
             left=top_risk['cold_risk_pct'],
             color='#D94A4A', label='Heat risk (≥35°C)', height=0.6)

axes[0].set_title('Thermal Risk Exposure by Sensor Location\n% of Readings in Dangerous Temperature Ranges',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('% of Total Readings', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

for i, (h, c) in enumerate(zip(top_risk['heat_risk_pct'], top_risk['cold_risk_pct'])):
    if h > 0.5:
        axes[0].text(c + h/2, i, f'{h}%', va='center', ha='center', fontsize=8, color='white', fontweight='bold')
    if c > 0.5:
        axes[0].text(c/2, i, f'{c}%', va='center', ha='center', fontsize=8, color='white', fontweight='bold')

y_pos = range(len(top_risk))
axes[1].hlines(y_pos, top_risk['min_temp'], top_risk['max_temp'], color='#CCCCCC', linewidth=2)
axes[1].scatter(top_risk['min_temp'], y_pos, color='#4A90D9', s=60, zorder=3, label='Min temp')
axes[1].scatter(top_risk['avg_temp'], y_pos, color='#F5A623', s=60, zorder=3, label='Avg temp')
axes[1].scatter(top_risk['max_temp'], y_pos, color='#D94A4A', s=60, zorder=3, label='Max temp')
axes[1].axvline(x=HEAT_THRESHOLD, color='#D94A4A', linestyle='--', alpha=0.5, label=f'Heat threshold ({HEAT_THRESHOLD}°C)')
axes[1].axvline(x=COLD_THRESHOLD, color='#4A90D9', linestyle='--', alpha=0.5, label=f'Cold threshold ({COLD_THRESHOLD}°C)')

axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(top_risk['short_name'], fontsize=9)
axes[1].set_title('Temperature Range per Sensor Location\n(Min / Average / Max)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Temperature (°C)', fontsize=11)
axes[1].legend(fontsize=9, loc='lower right')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("\nTop 5 locations by total thermal risk exposure:")
print(top_risk[['short_name','heat_risk_pct','cold_risk_pct','total_risk_pct','avg_temp']].head(5).to_string(index=False))

### 7.2 Output Analysis- Thermal Risk Profile

**Thermal Risk Exposure (Stacked Bar Chart)**
- **Royal Park Asset ID: COM2707** is a significant outlier recording 4.9% cold risk readings, far exceeding every other location. This sensor experiences genuinely dangerous cold conditions far more frequently than any CBD location, making it a critical priority for overnight outreach during winter months
- **Birrarung Marr Park** ranks second with a combined risk of 2.7% - the highest among recognisable CBD locations. As an open riverside park with limited shelter, this is exactly the kind of exposed location where rough sleepers are most vulnerable overnight
- **Tram Stop 7B and 7C** (Melbourne Tennis Centre Precinct) show combined risk of 1.8% and 2.1% respectively - transport infrastructure locations that may attract rough sleepers seeking covered shelter but offer little thermal protection
- **Swanston Street** appears in the top 5 with 1.6% combined risk, notable because it also appeared as one of the busiest overnight pedestrian locations, confirming it as a high-exposure, high-risk zone
- Rooftop sensors (CH1, SkyFarm, 101 Collins St L11) show predominantly heat risk -their elevated position increases solar exposure during the day, but these locations are inaccessible to rough sleepers and are less relevant to ground-level risk

**Temperature Range Dot Plot**
- All 12 locations share a strikingly similar pattern -minimum temperatures cluster just above 0°C (blue dots), average temperatures sit between 13-17°C (orange dots), and maximum temperatures all breach the 35°C heat threshold (red dots extending past the red dashed line)
- This confirms that dangerous heat exposure is **universal across all sensor locations** - no CBD precinct is insulated from summer heat spikes
- **Royal Park** is the only location where the minimum temperature (blue dot) sits at or below the cold threshold (blue dashed line at 5°C) - reinforcing its status as the highest cold-risk location in the dataset
- The consistent gap between average and maximum temperatures across all locations indicates that heat events are acute and episodic rather than sustained-meaning outreach timing during heatwave days is critical.

**Key NightWatch Insight**
Royal Park and Birrarung Marr emerge as the two locations with the most dangerous thermal profiles - both are open, exposed outdoor spaces with limited shelter infrastructure, making them high-priority targets for the Survival Risk Surface scoring model.